## Multi-task auxiliary objectives update
This notebook has been updated to work with the extended multi-task ASR head. You can enable or disable CTC, seq2seq, frame-level phoneme classifiers, speaker embeddings, and pronunciation error classifiers individually through `Configs/config.yml` under the `multi_task` section. Adjust the corresponding `loss_weights` entries when experimenting with these objectives.

> **Entropy regularisation**: Use the new `regularization.entropy` section in `Configs/config.yml` to enable or disable per-head entropy penalties. You can independently attach the regulariser to the CTC and seq2seq objectives without breaking joint training.

## Memory optimization settings
Lazy creation of decoder masks can now be toggled through the new `memory_optimizations.lazy_masks` section in `Configs/config.yml`. When enabled (the default), the trainer skips preallocating the `future_mask` and `text_mask` tensors unless an experiment explicitly needs them. Set the corresponding flags to `false` to restore the previous eager allocation behaviour.


# AuxiliaryASR Duration Statistics

This notebook loads a trained AuxiliaryASR model, prepares the validation dataset, and computes duration statistics for the CTC best-path decoding.

## Augmentation configuration
This notebook now honours the extended SpecAugment policies, waveform perturbations, mixup, and phoneme-level dropout toggles defined in `Configs/config.yml`. Adjust those settings in the config file before running these evaluation cells.


In [ ]:
# check available CUDA devices
import torch
devices = []
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        device_name = torch.cuda.get_device_name(i)
        devices.append({
            'type': 'CUDA',
            'available': True,
            'name': device_name,
            'index': i
        })
else:
    devices.append({'type': 'CUDA', 'available': False, 'name': 'N/A'})
devices

In [ ]:
# change folder into the root of the ASR project
import os

if not os.path.isdir('Configs'):
    %cd ../

!pwd

## Imports and helper utilities

In [ ]:
# import packages and define helper utilities
import os
import yaml
import torch
import pandas as pd
import numpy as np

from models import ASRCNN
from meldataset import build_dataloader
from token_map import build_token_map_from_data


def cfg_get_nested(cfg: dict, path, default=None, sep='.'):
    """Get a nested value from a dict using a list of keys or a dot-separated string."""
    if isinstance(path, str):
        keys = path.split(sep) if path else []
    else:
        keys = path

    cur = cfg
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur


def load_token_map_from_config(config):
    token_src = config.get('phoneme_maps_path')
    if not token_src:
        return build_token_map_from_data(
            config.get('train_data'),
            config.get('val_data'),
            config.get('ood_data'),
            apply_asr_tokenizer=True,
        )
    if isinstance(token_src, dict):
        return token_src
    csv = pd.read_csv(token_src, header=None).values
    return {word: index for word, index in csv}


def load_asr_model(model_path, config_path, device):
    with open(config_path) as f:
        config = yaml.safe_load(f)

    token_map = load_token_map_from_config(config)

    model_params = cfg_get_nested(config, 'model_params', {
        'input_dim': 80,
        'hidden_dim': 256,
        'n_token': len(token_map),
        'token_embedding_dim': 512,
        'n_layers': 5,
        'location_kernel_size': 31,
    })
    if 'n_token' not in model_params:
        model_params['n_token'] = len(token_map)

    model_params.setdefault('stabilization_config', cfg_get_nested(config, 'stabilization', {}))

    model = ASRCNN(**model_params)
    checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)
    state_dict = checkpoint.get('model', checkpoint)
    try:
        model.load_state_dict(state_dict)
    except RuntimeError:
        sanitized_state = {k.replace('module.', ''): v for k, v in state_dict.items()}
        model.load_state_dict(sanitized_state)

    model.to(device)
    model.eval()
    return model, config, token_map


def build_dev_dataloader(config, device, batch_size=None, num_workers=None):
    val_csv_path = config.get('val_data')
    if val_csv_path is None:
        raise ValueError("Validation CSV path ('val_data') not found in the config.")

    with open(val_csv_path, 'r', encoding='utf-8') as f:
        raw_lines = [line.rstrip('\n') for line in f]

    path_list = []
    for raw in raw_lines:
        if not raw.strip():
            continue
        parts = raw.split('|')
        if len(parts) == 1:
            continue
        path = parts[0]
        if len(parts) == 2:
            text = parts[1]
            speaker = ''
        else:
            text = '|'.join(parts[1:-1])
            speaker = parts[-1]
        path_list.append([path, text, speaker])

    if batch_size is None:
        batch_size = int(cfg_get_nested(config, 'eval_params.batch_size', cfg_get_nested(config, 'batch_size', 4)))
    if num_workers is None:
        num_workers = int(cfg_get_nested(config, 'dataloader_params.val_num_workers', 2))

    dataset_params = resolve_dataset_params(config)

    collate_config = {'return_wave': False}
    device_flag = device.type if isinstance(device, torch.device) else str(device)
    loader = build_dataloader(
        path_list=path_list,
        validation=True,
        batch_size=batch_size,
        num_workers=num_workers,
        device=device_flag,
        collate_config=collate_config,
        dataset_config=dataset_params,
    )
    return loader

memory_opts = config.get("memory_optimizations", {}) if isinstance(config, dict) else {}
lazy_mask_cfg = memory_opts.get("lazy_masks", {}) if isinstance(memory_opts, dict) else {}
lazy_enabled = bool(lazy_mask_cfg.get("enabled", True))
print(f"lazy mask creation enabled --> {lazy_enabled}")
print(f"skip future mask allocation --> {bool(lazy_mask_cfg.get('future_mask', True))}")
print(f"skip text mask allocation --> {bool(lazy_mask_cfg.get('text_mask', True))}")


In [ ]:
def resolve_dataset_params(config, base_overrides=None):
    dataset_params = {
        'dict_path': cfg_get_nested(config, 'phoneme_maps_path', 'Data/word_index_dict.txt'),
        'sr': cfg_get_nested(config, 'preprocess_params.sr', cfg_get_nested(config, 'preprocess_parasm.sr', 24000)),
        'spect_params': cfg_get_nested(
            config,
            'preprocess_params.spect_params',
            cfg_get_nested(config, 'preprocess_parasm.spect_params', {'n_fft': 1024, 'win_length': 1024, 'hop_length': 300}),
        ),
        'mel_params': cfg_get_nested(
            config,
            'preprocess_params.mel_params',
            cfg_get_nested(config, 'preprocess_parasm.mel_params', {'n_mels': 80}),
        ),
    }
    dataset_overrides = cfg_get_nested(config, 'dataset_params', {})
    if isinstance(dataset_overrides, dict):
        for key in ('dict_path', 'sr', 'spect_params', 'mel_params'):
            if key in dataset_overrides:
                dataset_params[key] = dataset_overrides[key]
        if 'spec_augment' in dataset_overrides:
            dataset_params['spec_augment_params'] = dataset_overrides['spec_augment']
        for key, value in dataset_overrides.items():
            if key in ('dict_path', 'sr', 'spect_params', 'mel_params', 'spec_augment'):
                continue
            dataset_params[key] = value
    if base_overrides:
        dataset_params.update(base_overrides)
    return dataset_params



## Load model, configuration, and validation loader

In [ ]:
checkpoint_dir = 'Checkpoint'
config_path = 'Checkpoint/config.yml'

if not os.path.isdir(checkpoint_dir):
    raise FileNotFoundError(f"Checkpoint directory '{checkpoint_dir}' not found.")

ckpt_files = [f for f in os.listdir(checkpoint_dir) if f.startswith('epoch_') and f.endswith('.pth')]
if not ckpt_files:
    raise FileNotFoundError(f"No checkpoint files found in '{checkpoint_dir}'.")

ckpt_files = sorted(ckpt_files, key=lambda x: int(x.split('_')[-1].split('.')[0]))
model_path = os.path.join(checkpoint_dir, ckpt_files[-1])
print(f"Loading model from {model_path}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, config, token_map = load_asr_model(model_path, config_path, device)

print(f'model --> {model_path}')
print(f'config --> {config_path}')
phoneme_source = config.get('phoneme_maps_path', 'built from dataset')
print(f'dictionary --> {phoneme_source}')

dev_loader = build_dev_dataloader(config, device)
print(f'Validation dataset size: {len(dev_loader.dataset)} samples')

if ' ' not in token_map:
    raise KeyError("The vocabulary does not contain the blank symbol ' '.")
BLANK_ID = token_map[' ']
print(f'Blank token id: {BLANK_ID}')

memory_opts = config.get("memory_optimizations", {}) if isinstance(config, dict) else {}
lazy_mask_cfg = memory_opts.get("lazy_masks", {}) if isinstance(memory_opts, dict) else {}
lazy_enabled = bool(lazy_mask_cfg.get("enabled", True))
print(f"lazy mask creation enabled --> {lazy_enabled}")
print(f"skip future mask allocation --> {bool(lazy_mask_cfg.get('future_mask', True))}")
print(f"skip text mask allocation --> {bool(lazy_mask_cfg.get('text_mask', True))}")


## Duration statistics computation
- frames_per_token_mean should be consistent with your hop size and speaking rate (e.g., hop 300 @ 24 kHz ≈ 12.5 ms/frame → 8–15 frames/phone is typical for EN; tonal languages can vary).
- extremely short (1–2 frame) phones dominating or huge tails (>200 frames) often indicate alignment issues.

In [ ]:
# compute duration statistics using CTC best-path decoding
@torch.no_grad()
def best_path_durations(logits, lens, blank_id):
    ids_all, durs_all = [], []
    path = logits.argmax(-1)
    for b in range(path.size(0)):
        prev = None
        ids_b, durs_b = [], []
        T = int(lens[b])
        for t in range(T):
            cur = int(path[b, t])
            if cur == blank_id:
                prev = cur
                continue
            if prev != cur:
                ids_b.append(cur)
                durs_b.append(1)
            else:
                durs_b[-1] += 1
            prev = cur
        ids_all.append(ids_b)
        durs_all.append(durs_b)
    return ids_all, durs_all


@torch.no_grad()
def duration_stats(model, dev_loader, device=None, blank_id=0):
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    all_durs = []
    ratio_frames_to_tokens = []
    downsample_factor = 2 ** getattr(model, 'n_down', 1)

    for batch in dev_loader:
        texts, text_lens, mels, mel_lens = batch[:4]
        mels = mels.to(device)
        mel_lens = mel_lens.to(torch.long)

        outputs = model(mels)
        if isinstance(outputs, dict):
            logits = outputs.get('logits_ctc')
            if logits is None:
                raise KeyError("Model output dict does not contain 'logits_ctc'.")
        elif isinstance(outputs, (tuple, list)):
            logits = outputs[0]
        else:
            logits = outputs

        logits = logits.detach().cpu()
        logit_lens = torch.clamp(mel_lens // downsample_factor, min=1, max=logits.size(1)).cpu()
        ids, durs = best_path_durations(logits, logit_lens, blank_id)

        mel_lens_cpu = mel_lens.cpu()
        for mel_len, dur in zip(mel_lens_cpu, durs):
            if len(dur) == 0:
                continue
            ratio_frames_to_tokens.append(float(mel_len.item()) / len(dur))
            all_durs.extend(dur)

    if not all_durs:
        raise RuntimeError('No non-blank token durations collected; check the model outputs and data.')

    all_durs = np.array(all_durs, dtype=np.float32)
    ratio_frames_to_tokens = np.array(ratio_frames_to_tokens, dtype=np.float32)

    return {
        'mean_dur_frames': float(all_durs.mean()),
        'p50': float(np.percentile(all_durs, 50)),
        'p90': float(np.percentile(all_durs, 90)),
        'frames_per_token_mean': float(ratio_frames_to_tokens.mean()),
    }


stats = duration_stats(model, dev_loader, device=device, blank_id=BLANK_ID)
print(stats)


## Self-Conditioned CTC Support

This notebook has been updated to work with the optional self-conditioned CTC feature. To experiment with it, enable the feature in `Configs/config.yml` by setting:

```yaml
stabilization:
  self_conditioned_ctc:
    enabled: true
    layers:
      - index: 2
      - index: 4
    conditioning_strategy: add  # or concat
    detach_conditioning: true
loss_weights:
  self_conditioned_ctc: 0.2
```

With the feature enabled the trainer will expose `self_conditioned_ctc_logits` and `self_conditioned_ctc_log_probs` in the model outputs so they can be visualised or scored inside the notebook just like the existing CTC signals.

```yaml
regularization:
  entropy:
    enabled: false
    mode: minimize  # set to "maximize" to encourage smoother distributions
    eps: 1.0e-6
    targets:
      ctc:
        enabled: true
        weight: 0.0
        length_normalize: true
      s2s:
        enabled: true
        weight: 0.0
        length_normalize: true
```